# Lab 04 solution

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("../../data")
OUT = Path("output")
OUT.mkdir(exist_ok=True)

In [ ]:
trade = pd.read_csv(DATA / "trade_summary.csv")
tariffs = pd.read_csv(DATA / "tariffs_mfn.csv")
countries = pd.read_excel(DATA / "countries.xlsx", sheet_name="countries")

In [ ]:
# check
assert trade.shape == (280, 8)
assert tariffs.shape == (280, 4)
assert len(countries) == 14
print("Part A OK")

In [ ]:
for name, df in [("trade", trade), ("tariffs", tariffs), ("countries", countries)]:
    print("=" * 20, name)
    df.info()
    print("Missing:\n", df.isna().sum())
    print("Duplicates:", df.duplicated().sum())
    print(df.describe().round(1))

print(trade["region"].value_counts())
print(trade["product_group"].value_counts())

**Findings:** `trade` and `countries` are complete with no duplicates or negative values. `tariffs` has a small number of missing `avg_mfn_tariff_pct` values, so any averages will silently skip them. Each reporter has 20 rows (5 years x 4 products), so the panel is balanced. Values are nominal USD millions.

In [ ]:
kenya = trade.loc[trade["reporter"] == "Kenya", ["year", "product_group", "exports_usd_m"]]

manuf_2023 = (trade[(trade["product_code"] == "MAN") & (trade["year"] == 2023)]
              .sort_values("exports_usd_m", ascending=False))

print((trade["imports_usd_m"] > 1_000_000).sum(), "rows with imports above USD 1 trillion")

In [ ]:
# check
assert list(kenya.columns) == ["year", "product_group", "exports_usd_m"]
assert len(manuf_2023) == 14
assert manuf_2023.iloc[0]["reporter"] == "China"
print("Part C OK")

In [ ]:
trade["balance_usd_m"] = trade["exports_usd_m"] - trade["imports_usd_m"]
trade["exports_usd_bn"] = (trade["exports_usd_m"] / 1000).round(1)
reporter_total = trade.groupby(["reporter", "year"])["exports_usd_m"].transform("sum")
trade["share_of_reporter_pct"] = trade["exports_usd_m"] / reporter_total * 100
trade.head()

In [ ]:
# check
shares = trade.groupby(["reporter", "year"])["share_of_reporter_pct"].sum().round(6)
assert (shares == 100).all()
print("Part D OK")

In [ ]:
region_year = trade.pivot_table(index="region", columns="year", values="exports_usd_m", aggfunc="sum").round(0)

top5_2023 = (trade[trade["year"] == 2023].groupby("reporter")["exports_usd_m"].sum()
             .sort_values(ascending=False).head(5))

totals = trade.pivot_table(index="reporter", columns="year", values="exports_usd_m", aggfunc="sum")
growth = ((totals[2023] / totals[2019] - 1) * 100).round(1).sort_values(ascending=False)

t23 = trade[trade["year"] == 2023]
agr_share = (t23[t23["product_code"] == "AGR"].set_index("reporter")["share_of_reporter_pct"]
             .round(1).sort_values(ascending=False))

display(region_year, top5_2023, growth, agr_share)

In [ ]:
# check
assert region_year.shape == (5, 5)
assert top5_2023.index[0] == "China"
assert len(growth) == 14
print("Part E OK")

In [ ]:
merged = trade.merge(countries[["iso3", "income_group"]], left_on="reporter_iso3", right_on="iso3", how="left")
print(merged[merged["year"] == 2023].groupby("income_group")["exports_usd_m"].sum().round(0))

with_tariffs = merged.merge(tariffs, on=["iso3", "year", "product_code"], how="left")
agr = with_tariffs[with_tariffs["product_code"] == "AGR"]
agr.groupby("income_group")["avg_mfn_tariff_pct"].mean().round(2)

In [ ]:
with pd.ExcelWriter(OUT / "day1_summary.xlsx") as writer:
    region_year.to_excel(writer, sheet_name="region_year")
    top5_2023.to_excel(writer, sheet_name="top5_2023")
    growth.to_excel(writer, sheet_name="growth")
    agr_share.to_excel(writer, sheet_name="agr_share")

In [ ]:
# check
sheets = pd.ExcelFile(OUT / "day1_summary.xlsx").sheet_names
assert {"region_year", "top5_2023", "growth", "agr_share"} <= set(sheets)
print("Part G OK")

**Example findings** (illustrative data):
- Exports in every region dipped in 2020 and exceeded 2019 levels by 2021-2022.
- China is by far the largest exporter, around double the next economy.
- Agricultural products make up a noticeably larger share of exports for some lower middle income economies, which matters for tariff discussions on agriculture.